In [5]:
# Önce kurulum (ilk çalıştırmada bir kez)
# !pip install requests beautifulsoup4 pandas

import re
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote, urljoin

kategori = input("Kategori adını girin (ör: telefon, laptop, ayakkabı): ").strip()

def review_sayisi_cek(html):
    patterns = [
        r'"reviewCount"\s*:\s*(\d+)',
        r'"review_count"\s*:\s*(\d+)',
        r'"ratingCount"\s*:\s*(\d+)',
        r'"reviewCount"\s*:\s*"(\d+)"',
        r'(\d+)\s*değerlendirme',
        r'(\d+)\s*yorum',
        r'(\d+)\s*Yorum',
    ]

    for pat in patterns:
        m = re.search(pat, html, re.I)
        if m:
            return int(re.sub(r"\D", "", m.group(1)))

    return None

def trendyol_urunleri(kategori, limit=10):
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    }

    url = f"https://www.trendyol.com/sr?q={quote(kategori)}"
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    urun_linkleri = []
    seen = set()

    for a in soup.select("a[href]"):
        href = a.get("href", "")
        if not href:
            continue

        if "p-" not in href:
            continue

        if href.startswith("/"):
            href = urljoin("https://www.trendyol.com", href)

        if href in seen:
            continue

        seen.add(href)

        title = " ".join(a.get_text(" ", strip=True).split())
        if len(title) < 3:
            continue

        urun_linkleri.append((title, href))

        if len(urun_linkleri) >= limit:
            break

    results = []

    for title, href in urun_linkleri:
        try:
            pr = requests.get(href, headers=headers, timeout=20)
            pr.raise_for_status()
            count = review_sayisi_cek(pr.text)
            results.append({
                "ürün": title,
                "link": href,
                "değerlendirme_sayısı": count
            })
        except Exception:
            results.append({
                "ürün": title,
                "link": href,
                "değerlendirme_sayısı": None
            })

    return pd.DataFrame(results)

df = trendyol_urunleri(kategori, limit=10)

print(f"\nKategori: {kategori}\n")
print(df.to_string(index=False))

def marka_cek(urun_adi):
    if not isinstance(urun_adi, str):
        return ""
    parcalar = urun_adi.split()
    return parcalar[0] if parcalar else ""

df["marka"] = df["ürün"].apply(marka_cek)
markalar = [m for m in df["marka"].tolist() if m]
markalar = list(dict.fromkeys(markalar))

print("\nMarkalar:")
for marka in markalar:
    print(marka)


Kategori: kokteyl seti

                                                                                                                                                                                                                                    ürün                                                                                                                                                                                    link  değerlendirme_sayısı
          Hızlı Bakış En Çok Satan 1. Ürün vosco 6 Parça Avantajlı Kokteyl Set Boston Shaker (750ml) 4.7 ( 29 ) Sepette %1 İndirim Kargo Bedava Kupon Fırsatı Sepette %1 İndirim Sepette 937,58 TL 947,05 TL Trendyol Plus ile 887,58 TL                                                            https://www.trendyol.com/vosco/6-parca-avantajli-kokteyl-set-boston-shaker-750ml-p-786453276?boutiqueId=61&merchantId=324262                    17
Hızlı Bakış En Çok Favorilenen 1. Ürün Bar Boss 10 Parça Süper Avantajlı Paket Pipetli Kokteyl Ha